In [19]:
# ====================================================
# NEAR MISS AUDIT REPORT - 2011
# SHEET1 CLEANING
# ====================================================

import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

# ====================================================
# LOAD
# ====================================================

file = "../2. Near Miss Audit Report - 2011 New Format.xlsx"
sheet = "Sheet1"

df = pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

original_shape = df.shape

print("Loaded:", sheet)
print("Original Shape:", original_shape)

# ====================================================
# CLEAN COLUMN NAMES
# ====================================================

def clean(col):

    col = str(col).strip()

    col = re.sub(
        r"[^\w]+",
        "_",
        col
    )

    col = re.sub(
        "_+",
        "_",
        col
    )

    return col.strip("_")


df.columns = [
    clean(c)
    for c in df.columns
]

# ====================================================
# CLEAN TEXT (KEEP REF_NO EXACTLY)
# ====================================================

obj_cols = df.select_dtypes(
    include="object"
).columns

for col in obj_cols:

    if "ref" in col.lower():

        df[col] = (

            df[col]

            .astype("string")

            .replace(
                r"^\s*$",
                np.nan,
                regex=True
            )

            .str.strip()

        )

    else:

        df[col] = (

            df[col]

            .astype("string")

            .replace(
                r"^\s*$",
                np.nan,
                regex=True
            )

            .str.strip()

        )
# ====================================================
# FIX REF_NO DATE CONVERSION
# ====================================================

ref_cols = [
    c
    for c in df.columns
    if "ref" in c.lower()
]

def restore_ref(v):

    if pd.isna(v):
        return np.nan

    v = str(v).strip()

    try:

        dt = pd.to_datetime(
            v,
            errors="raise"
        )

        if (
            len(v) > 8
            and ":" in v
        ):

            return dt.strftime(
                "%d-%b"
            )

    except:
        pass

    return v


for col in ref_cols:

    df[col] = (

        df[col]

        .astype("string")

        .str.strip()

        .replace(
            r"^\s*$",
            np.nan,
            regex=True
        )

        .apply(
            restore_ref
        )

    )

# ====================================================
# FORMAT DATE COLUMNS → 12-Oct-10
# ====================================================

date_cols = [

    c

    for c in df.columns

    if "date" in c.lower()

]

for col in date_cols:

    df[col] = (

        pd.to_datetime(
            df[col],
            errors="coerce",
            dayfirst=True
        )

        .dt.strftime(
            "%d-%b-%y"
        )

        .fillna(
            df[col]
        )

    )
# ====================================================
# REMOVE 100% EMPTY COLUMNS
# KEEP SI NO
# ====================================================

serial_col = None

for c in df.columns:

    if (
        "sl" in c.lower()
        or
        "si" in c.lower()
    ):

        serial_col = c
        break

remove_cols = []

for col in df.columns:

    if col == serial_col:
        continue

    if df[col].isna().all():

        remove_cols.append(col)

df.drop(
    columns=remove_cols,
    inplace=True
)


# ====================================================
# CREATE SERIAL NUMBER
# ====================================================

if serial_col is None:

    serial_col = "SI_No"

    df.insert(
        0,
        serial_col,
        range(1, len(df)+1)
    )

else:

    df[serial_col] = range(
        1,
        len(df)+1
    )

# ====================================================
# REMOVE FULL EMPTY ROWS
# ====================================================

before = len(df)

df.dropna(
    how="all",
    inplace=True
)

rows_removed = before - len(df)

# ====================================================
# CHECK DUPLICATES
# ====================================================

duplicates = df.duplicated().sum()

duplicate_rows = df[
    df.duplicated()
]

df.drop_duplicates(
    inplace=True
)

# ====================================================
# MISSING VALUES
# ====================================================

missing = (
    df
    .isna()
    .sum()
)

missing = missing[
    missing > 0
]

# ====================================================
# FINAL REPORT
# ====================================================

print("\nColumns After Removing 100% Missing:")
print(df.columns.tolist())

print("\nDuplicate Rows Found:")
print(duplicates)

if duplicates > 0:
    print(duplicate_rows.head())

print("\nMissing Values Summary:")
print(missing)

print("\nRows Removed:", rows_removed)

print("\nFinal Shape:")
print(df.shape)

# ====================================================
# EXPORT
# ====================================================

output = "cleaned_2011_sheet1_1234.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        index=False
    )

    ws = writer.sheets["Sheet1"]

    for col in ws.columns:

        width = max(

            len(
                str(cell.value)
            )

            if cell.value

            else 0

            for cell

            in col

        )

        ws.column_dimensions[
            col[0].column_letter
        ].width = min(
            width + 3,
            80
        )

print("\nSaved:", output)

Loaded: Sheet1
Original Shape: (24, 20)

Columns After Removing 100% Missing:
['Sl_No', 'Ref_No', 'Name_of_Ship', 'Type_of_Ship', 'Date_of_the_Near_Miss', 'Description_of_the_incident_and_cause_analysis', 'Incident_Category_Potential', 'Probability_of_Reoccurence', 'Cause_Analysis', 'Counter_measure_to_prevent_recurrence']

Duplicate Rows Found:
0

Missing Values Summary:
Ref_No                         1
Incident_Category_Potential    1
Probability_of_Reoccurence     3
Cause_Analysis                 1
dtype: int64

Rows Removed: 0

Final Shape:
(24, 10)

Saved: cleaned_2011_sheet1_1234.xlsx
